# Five stages, and other multi-step scoring

Rodent three-stage scoring and human five-stage scoring are **the same code**.
They differ in how many clustering steps run and what the clusters are called —
configuration, not a different pipeline.

A step takes the epochs that currently carry one label, fits a PCA *inside just
those epochs*, clusters them, and replaces that label with finer ones. Going
finer is a matter of adding a step.

This notebook needs a real human recording. It is written against the Dreem Open
Datasets, which nyx has a recipe for; section 1 says how to point it elsewhere.
For something you can run with no data, see `01_score_recording.ipynb`.

## 1. The data

The Dreem Open Datasets (DOD-H, DOD-O) are freely available for research from
<https://github.com/Dreem-Organization/dreem-learning-open>. Download them into a
folder laid out as the recipe expects:

```
<data_root>/dodh/recordings/<uuid>.edf
<data_root>/dodh/annotations/<uuid>/scorer_1.csv
```

Then set `DATA_ROOT` and `RECORDING` below.

**Any human PSG works** — the recipe is a convenience, not a requirement. For
your own file:

```python
recording = nyx.read_recording(path, eeg_channel="C3_M2", emg_channel="EMG")
reference = nyx.read_annotations(scores, format="epoch_csv", epoch_length=30,
                                 label_map={0: "WAKE", 1: "NREM1", 2: "NREM2",
                                            3: "NREM3", 4: "REM"})
```

`nyx.datasets.list_datasets()` shows the recipes that exist.

In [ ]:
import os

for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = "1"

import matplotlib.pyplot as plt
import pandas as pd

import nyx
from nyx.datasets import list_datasets, load_dataset

list_datasets()

In [ ]:
DATA_ROOT = r"...\data"                 # <- where you downloaded it
RECORDING = "0d79f4b1-e74f-5e87-8e42-f9dd7112ada5"   # <- a uuid from recordings/

recording, reference = load_dataset("dodh", data_root=DATA_ROOT,
                                    recording_name=RECORDING)
print(recording.describe())

## 2. What the steps say

`params/human.json` reaches five stages in four steps. Read it as a sequence of
subdivisions:

| step | takes | produces |
|---|---|---|
| `no_emg_split` | — | everything → SLEEP |
| `wake_sleep` | SLEEP | WAKE / SLEEP |
| `split_wake` | WAKE | WAKE / NREM1 |
| `split_sleep` | SLEEP | NREM2 / NREM3 / REM |

The first step is an `emg_threshold` with a deliberately **permissive** cut, so
nothing is called wake on EMG power alone and everything lands in SLEEP. Human
recordings are short and quiet — lying still awake looks like sleep to an EMG —
so wake is better recovered from the EEG, which the second step does.

That second step uses `elliptic`: it fits a robust Gaussian to the bulk of the
data and calls the tail outliers. Sleep is one dense blob and wake is the
scattered minority, so a method that models "the bulk plus everything else"
beats one assuming two comparable clusters.

In [ ]:
params = nyx.load_params("../params/human.json")

for step in params["steps"]:
    shown = {k: v for k, v in step.items() if not k.startswith("_")}
    print(shown)
    print()

## 3. Score it

`score_recording` runs the whole list. This is the same call as for a rodent —
only the params differ.

In [ ]:
result = nyx.score_recording(recording, params, reference=reference, verbose=True)

`result.steps` holds one outcome per clustering step, so you can see what each
one decided rather than only the final answer.

In [ ]:
for outcome in result.steps:
    print(outcome.summary())
    print()

## 4. 🔧 These parameters are a starting point

`human.json` is the most common configuration across the runs in the paper, but
human scoring **needs per-recording attention** in a way rodent scoring usually
does not. Two things to check on every recording:

- **The per-cluster spectra.** NREM3 should carry the most low-frequency power,
  REM the least. If the ordering is wrong, fix `stage_order` or set
  `cluster_to_stage` on that step.
- **`elliptic_contamination`** on the wake/sleep step — how much of the
  recording is treated as the wake tail. The default 0.1 suits a night with
  about 10% wake; a restless night needs more.

To change a step, edit the params and re-run:

```python
params["steps"][1]["elliptic_contamination"] = 0.15
params["steps"][3]["stage_order"] = ["NREM3", "NREM2", "NREM2", "REM"]
result = nyx.score_recording(recording, params, reference=reference)
```

Mapping every cluster of a step to the same stage makes that step a no-op, which
is the natural way to decide — after looking at it — that a split was not worth
keeping.

In [ ]:
nyx.plot_summary(result)
plt.show()

## 5. Coarser hypnograms come for free

Scoring runs once at the finest granularity it can reach. Coarser versions are
then merges of that one, so they are guaranteed consistent with each other —
rather than three separate scorings that might disagree.

- **4-stage** folds N1 into N2. They are the hardest pair for human scorers to
  separate, and N1 is a small fraction of the night, so four stages is often the
  fairer comparison.
- **3-stage** merges all NREM. This is the vocabulary rodent scoring uses, and
  what lets a human run be compared against a rodent-style reference.

In [ ]:
from nyx.granularity import available_granularities

print("available:", available_granularities(result.hypnogram))

for n in available_granularities(result.hypnogram):
    collapsed = nyx.collapse(result.hypnogram, n)
    stages = sorted(set(collapsed["label"]) - {"NOSIGNAL"})
    agreement = nyx.evaluate(collapsed, reference, window=result.window,
                             label_order=stages, verbose=False)
    print(f"  {n}-stage  MF1 {agreement.mf1:.3f}  accuracy {agreement.accuracy:.3f}"
          f"   {stages}")

Agreement rises as the vocabulary coarsens, which is exactly what you would
expect: the disagreements that disappear are the N1/N2 and N2/N3 boundaries,
which human scorers also disagree about. Report the granularity that matches
what you are comparing against, and say which one it is.

## 6. Rodent substages

The same mechanism, going the other way. `params/mouse_substages.json` adds a
third step that subdivides NREM into NREM2, NREM3 and TR (transition):

```jsonc
{ "name": "split_nrem", "within": "NREM", "method": "gmm", "n_clusters": 3,
  "stage_order": ["TR", "NREM3", "NREM2"] }
```

`gmm` with a fixed `n_clusters` rather than `hdbscan`, because the substages are
not separated by a density gap the way NREM and REM are — letting the algorithm
choose how many there are gives a different count on every recording.

`nyx.collapse(hypnogram, 3)` folds them back, so a substage run stays comparable
against an ordinary three-stage rodent reference. It will not be *identical* to a
two-step run: the extra step brings its own outlier rejection, so a few more
epochs come out as NOSIGNAL.

**That `stage_order` is not a safe default.** The sign of a principal component
is arbitrary — check the per-cluster spectra and reorder if they disagree.

## 7. Save

In [ ]:
OUTPUT_DIR = f"../results/{RECORDING}"

nyx.save_results(result, OUTPUT_DIR, granularities=[4, 3])
print("saved to", os.path.abspath(OUTPUT_DIR))
print(sorted(os.listdir(OUTPUT_DIR)))

That writes `hypnogram.csv` at five stages plus `hypnogram_4stage.csv` and
`hypnogram_3stage.csv`, and a `run.json` carrying every step's resolved
cluster-to-stage mapping — the part that cannot be recovered from the params
alone.

Cluster ids are renumbered by centroid, so a saved mapping means the same thing
on every run. Without that, replaying saved parameters can attach stage names to
the wrong clusters.

---

For many recordings, `03_batch_queue.ipynb`. For what goes in a parameter file,
`params/README.md`.